# SilverAI — Feature Engineering

## Objective
- Engineer an initial candidate set of 17 technically sound, non-redundant features from cleaned 5-minute silver futures data (`silver_5m.csv`).
- Capture price momentum, volatility clustering, trend distance, candle geometry, volume dynamics, and cyclical time patterns without lookahead bias.

## Imports & Configuration
- Load libraries, define project filesystem paths, set up logging, and configure rolling window parameters.

In [1]:
import logging
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
INPUT_FILE = DATA_DIR / "silver_5m.csv"
OUTPUT_FILE = DATA_DIR / "silver_features.csv"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    force=True
)
logger = logging.getLogger("SilverAI.FeatureEngineering")

CONFIG = {
    "input_file": INPUT_FILE,
    "output_file": OUTPUT_FILE,
    "momentum_lags": [1, 3, 6],
    "volatility_windows": [5, 20, 50],
    "trend_windows": [20, 50],
    "volume_window": 20
}

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 130)

print(f"Project directory: {PROJECT_DIR}")
print(f"Input target:      {INPUT_FILE.name}")
print(f"Output target:     {OUTPUT_FILE.name}")

Project directory: C:\Users\ravik\Desktop\SilverAI
Input target:      silver_5m.csv
Output target:     silver_features.csv


## Load & Validate Data
- Ingest `data/silver_5m.csv`, parse Eastern Time timestamps, sort chronologically, and verify data integrity before engineering.

In [2]:
def load_and_validate_intraday(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing input dataset: {path}")
    df = pd.read_csv(path)
    df["datetime_et"] = pd.to_datetime(df["datetime_et"], utc=True).dt.tz_convert("America/New_York")
    df = df.sort_values("datetime_et").reset_index(drop=True)
    logger.info(f"Loaded {path.name}: {len(df):,} rows x {df.shape[1]} columns.")
    return df

df = load_and_validate_intraday(INPUT_FILE)

expected_cols = ["datetime_et", "open", "high", "low", "close", "volume"]
missing_cols = [c for c in expected_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Schema mismatch: missing {missing_cols}")

deltas = df["datetime_et"].diff().dropna()
pct_continuous = (deltas == pd.Timedelta(minutes=5)).mean() * 100

audit_table = pd.DataFrame([
    ("Observations", f"{len(df):,}"),
    ("Columns", f"{df.columns.tolist()}"),
    ("Date Range", f"{df['datetime_et'].min()} -> {df['datetime_et'].max()}"),
    ("Missing Values", f"{df.isnull().sum().sum()}"),
    ("Duplicate Timestamps", f"{df.duplicated(subset=['datetime_et']).sum()}"),
    ("Consecutive 5m Deltas", f"{pct_continuous:.2f}% (remaining are expected CME breaks)")
], columns=["Validation Check", "Status"])

display(audit_table)
display(df.head(3))

,Validation Check,Status
0,Observations,"189,798"
1,Columns,"['datetime_et', 'open', 'high', 'low', 'close'..."
2,Date Range,2023-11-01 02:10:00-04:00 -> 2026-11-03 02:00:...
3,Missing Values,0
4,Duplicate Timestamps,0
5,Consecutive 5m Deltas,93.13% (remaining are expected CME breaks)


,datetime_et,open,high,low,close,volume
0,2023-11-01 02:10:00-04:00,22.75,22.79,22.745,22.770,229
1,2023-11-01 02:15:00-04:00,22.77,22.79,22.765,22.785,84
2,2023-11-01 02:20:00-04:00,22.78,22.78,22.770,22.775,59


## Price Momentum Features
- Compute 1-candle (`return_1`), 3-candle (`return_3`), and 6-candle (`return_6`) percentage returns over sequential traded bars.
- Policy: Calculations use sequential observation indexing; opening bars reflect inter-session gap discovery without artificial interpolation.

In [3]:
def add_momentum_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["return_1"] = df["close"].pct_change(1) * 100.0
    df["return_3"] = df["close"].pct_change(3) * 100.0
    df["return_6"] = df["close"].pct_change(6) * 100.0
    logger.info("Added 3 momentum features: return_1, return_3, return_6.")
    return df

df = add_momentum_features(df)
display(df[["datetime_et", "close", "return_1", "return_3", "return_6"]].dropna().head(3))

,datetime_et,close,return_1,return_3,return_6
6,2023-11-01 02:45:00-04:00,22.765,0.000000,-0.021959,-0.021959
7,2023-11-01 02:50:00-04:00,22.745,-0.087854,-0.021978,-0.175554
8,2023-11-01 02:55:00-04:00,22.745,0.000000,-0.087854,-0.131723


## Volatility Features
- Compute rolling standard deviations of 5-minute returns over 5-candle (~25m), 20-candle (~100m), and 50-candle (~250m) observation windows.
- Windows are strictly backward-looking to ensure no future data leakage.

In [4]:
def add_volatility_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["volatility_5"] = df["return_1"].rolling(window=5).std()
    df["volatility_20"] = df["return_1"].rolling(window=20).std()
    df["volatility_50"] = df["return_1"].rolling(window=50).std()
    logger.info("Added 3 volatility features: volatility_5, volatility_20, volatility_50.")
    return df

df = add_volatility_features(df)
display(df[["datetime_et", "volatility_5", "volatility_20", "volatility_50"]].dropna().head(3))

,datetime_et,volatility_5,volatility_20,volatility_50
50,2023-11-01 06:35:00-04:00,0.024048,0.068690,0.080094
51,2023-11-01 06:45:00-04:00,0.032571,0.066053,0.079577
52,2023-11-01 07:00:00-04:00,0.092704,0.079206,0.084221


## Trend Features
- Calculate relative distance to 20-period and 50-period Simple Moving Averages: `(close / SMA) - 1`.
- Raw moving averages are kept internal and not exported to avoid absolute price-level non-stationarity.

In [5]:
def add_trend_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    sma_20 = df["close"].rolling(window=20).mean()
    sma_50 = df["close"].rolling(window=50).mean()
    df["sma_distance_20"] = (df["close"] / sma_20) - 1.0
    df["sma_distance_50"] = (df["close"] / sma_50) - 1.0
    logger.info("Added 2 trend distance features: sma_distance_20, sma_distance_50.")
    return df

df = add_trend_features(df)
display(df[["datetime_et", "close", "sma_distance_20", "sma_distance_50"]].dropna().head(3))

,datetime_et,close,sma_distance_20,sma_distance_50
49,2023-11-01 06:30:00-04:00,22.750,-0.000231,0.000625
50,2023-11-01 06:35:00-04:00,22.750,-0.000286,0.000642
51,2023-11-01 06:45:00-04:00,22.755,-0.000088,0.000889


## Candle Behaviour Features
- Extract intra-bar price action: signed return, total range relative to price, and upper/lower wick ratios.
- Zero-range candles ($High = Low$) are safely assigned `0.0` to eliminate division by zero.

In [6]:
def add_candle_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["candle_return"] = ((df["close"] - df["open"]) / df["open"]) * 100.0
    df["candle_range_pct"] = ((df["high"] - df["low"]) / df["close"]) * 100.0

    candle_range = df["high"] - df["low"]
    upper_wick = df["high"] - df[["open", "close"]].max(axis=1)
    lower_wick = df[["open", "close"]].min(axis=1) - df["low"]

    df["upper_wick_ratio"] = np.where(candle_range > 0, upper_wick / candle_range, 0.0)
    df["lower_wick_ratio"] = np.where(candle_range > 0, lower_wick / candle_range, 0.0)
    logger.info("Added 4 candle geometry features: candle_return, candle_range_pct, upper_wick_ratio, lower_wick_ratio.")
    return df

df = add_candle_features(df)
display(df[["datetime_et", "candle_return", "candle_range_pct", "upper_wick_ratio", "lower_wick_ratio"]].head(3))

,datetime_et,candle_return,candle_range_pct,upper_wick_ratio,lower_wick_ratio
0,2023-11-01 02:10:00-04:00,0.087912,0.197628,0.444444,0.111111
1,2023-11-01 02:15:00-04:00,0.065876,0.109721,0.200000,0.200000
2,2023-11-01 02:20:00-04:00,-0.021949,0.043908,0.000000,0.500000


## Volume Features
- Compute relative volume surge (`volume / SMA_20(volume)`) and bar-over-bar percentage volume change.
- Zero historical volume is guarded safely against division by zero.

In [7]:
def add_volume_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    vol_sma_20 = df["volume"].rolling(window=20).mean()
    df["relative_volume"] = np.where(vol_sma_20 > 0, df["volume"] / vol_sma_20, 0.0)

    prev_vol = df["volume"].shift(1)
    df["volume_change"] = np.where(prev_vol > 0, ((df["volume"] - prev_vol) / prev_vol) * 100.0, 0.0)
    logger.info("Added 2 volume features: relative_volume, volume_change.")
    return df

df = add_volume_features(df)
display(df[["datetime_et", "volume", "relative_volume", "volume_change"]].dropna().head(3))

,datetime_et,volume,relative_volume,volume_change
0,2023-11-01 02:10:00-04:00,229,0.0,0.000000
1,2023-11-01 02:15:00-04:00,84,0.0,-63.318777
2,2023-11-01 02:20:00-04:00,59,0.0,-29.761905


## Time-Based Features
- Encode diurnal intraday cycles using continuous harmonic encoding ($\sin / \cos$ of hour in ET) and calendar day of week.

In [8]:
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    hour_float = df["datetime_et"].dt.hour + df["datetime_et"].dt.minute / 60.0
    df["hour_sin"] = np.sin(2.0 * np.pi * hour_float / 24.0)
    df["hour_cos"] = np.cos(2.0 * np.pi * hour_float / 24.0)
    df["day_of_week"] = df["datetime_et"].dt.dayofweek
    logger.info("Added 3 time features: hour_sin, hour_cos, day_of_week.")
    return df

df = add_time_features(df)
display(df[["datetime_et", "hour_sin", "hour_cos", "day_of_week"]].head(3))

,datetime_et,hour_sin,hour_cos,day_of_week
0,2023-11-01 02:10:00-04:00,0.537300,0.843391,2
1,2023-11-01 02:15:00-04:00,0.555570,0.831470,2
2,2023-11-01 02:20:00-04:00,0.573576,0.819152,2


## Feature Dictionary
- Concise specification of all 17 candidate features, their calculation formulations, and analytical representations.

In [9]:
FEATURE_COLS = [
    "return_1", "return_3", "return_6",
    "volatility_5", "volatility_20", "volatility_50",
    "sma_distance_20", "sma_distance_50",
    "candle_return", "candle_range_pct", "upper_wick_ratio", "lower_wick_ratio",
    "relative_volume", "volume_change",
    "hour_sin", "hour_cos", "day_of_week"
]

feature_dict = pd.DataFrame([
    ("return_1", "Momentum", "close.pct_change(1) * 100", "Latest 5-minute price movement (%)"),
    ("return_3", "Momentum", "close.pct_change(3) * 100", "15-minute price movement (%)"),
    ("return_6", "Momentum", "close.pct_change(6) * 100", "30-minute price movement (%)"),
    ("volatility_5", "Volatility", "return_1.rolling(5).std()", "Short-term return volatility (~25 min)"),
    ("volatility_20", "Volatility", "return_1.rolling(20).std()", "Medium-term return volatility (~100 min)"),
    ("volatility_50", "Volatility", "return_1.rolling(50).std()", "Longer-term return volatility (~250 min)"),
    ("sma_distance_20", "Trend", "(close / SMA_20) - 1", "Relative distance of price to 20-bar trend"),
    ("sma_distance_50", "Trend", "(close / SMA_50) - 1", "Relative distance of price to 50-bar trend"),
    ("candle_return", "Candle", "(close - open) / open * 100", "Intra-candle directional return (%)"),
    ("candle_range_pct", "Candle", "(high - low) / close * 100", "Candle total range relative to price (%)"),
    ("upper_wick_ratio", "Candle", "(high - max(open, close)) / range", "Upper shadow wick proportion (0 to 1)"),
    ("lower_wick_ratio", "Candle", "(min(open, close) - low) / range", "Lower shadow wick proportion (0 to 1)"),
    ("relative_volume", "Volume", "volume / SMA_20(volume)", "Volume surge relative to 20-bar baseline"),
    ("volume_change", "Volume", "(volume - prev_vol) / prev_vol * 100", "Bar-over-bar volume change (%)"),
    ("hour_sin", "Time", "sin(2 * pi * hour / 24)", "Harmonic sine encoding of Eastern Time"),
    ("hour_cos", "Time", "cos(2 * pi * hour / 24)", "Harmonic cosine encoding of Eastern Time"),
    ("day_of_week", "Time", "datetime_et.dt.dayofweek", "Day of week index (Monday = 0, Sunday = 6)")
], columns=["Feature", "Category", "Calculation", "Description"])

display(feature_dict)

,Feature,Category,Calculation,Description
0,return_1,Momentum,close.pct_change(1) * 100,Latest 5-minute price movement (%)
1,return_3,Momentum,close.pct_change(3) * 100,15-minute price movement (%)
2,return_6,Momentum,close.pct_change(6) * 100,30-minute price movement (%)
3,volatility_5,Volatility,return_1.rolling(5).std(),Short-term return volatility (~25 min)
4,volatility_20,Volatility,return_1.rolling(20).std(),Medium-term return volatility (~100 min)
5,volatility_50,Volatility,return_1.rolling(50).std(),Longer-term return volatility (~250 min)
6,sma_distance_20,Trend,(close / SMA_20) - 1,Relative distance of price to 20-bar trend
7,sma_distance_50,Trend,(close / SMA_50) - 1,Relative distance of price to 50-bar trend
8,candle_return,Candle,(close - open) / open * 100,Intra-candle directional return (%)
9,candle_range_pct,Candle,(high - low) / close * 100,Candle total range relative to price (%)


## Feature Validation
- Audit missing values, infinite values, numerical ranges, and chronological monotonicity across all 17 engineered candidate features.

In [10]:
def validate_features(df: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    null_counts = df[feature_cols].isnull().sum()
    inf_counts = np.isinf(df[feature_cols]).sum()
    
    val_df = pd.DataFrame({
        "Missing": null_counts,
        "Inf": inf_counts,
        "Min": df[feature_cols].min().round(4),
        "Median": df[feature_cols].median().round(4),
        "Max": df[feature_cols].max().round(4),
        "Std": df[feature_cols].std().round(4)
    })
    val_df.index.name = "Feature"
    return val_df

validation_summary = validate_features(df, FEATURE_COLS)
display(validation_summary)

assert df["datetime_et"].is_monotonic_increasing, "Chronological order violation!"
assert np.isinf(df[FEATURE_COLS]).sum().sum() == 0, "Infinite values detected!"

max_nulls = df[FEATURE_COLS].isnull().sum().max()
print(f"\nWarmup Analysis: Exactly {max_nulls} initial rows contain warmup NaNs (from 50-period rolling lookback).")
print(f"Zero missing values exist from index 50 onward ({len(df) - max_nulls:,} complete feature observations).")


Warmup Analysis: Exactly 50 initial rows contain warmup NaNs (from 50-period rolling lookback).
Zero missing values exist from index 50 onward (189,748 complete feature observations).


,Missing,Inf,Min,Median,Max,Std
Feature,,,,,,
return_1,1,0,-4.5587,0.0000,6.2514,0.1741
return_3,3,0,-9.3613,0.0000,11.4329,0.3011
return_6,6,0,-11.0280,0.0083,12.8612,0.4251
volatility_5,5,0,0.0000,0.0931,4.0745,0.1201
volatility_20,20,0,0.0205,0.1050,2.6232,0.1092
volatility_50,50,0,0.0260,0.1130,1.7668,0.1029
sma_distance_20,19,0,-0.1215,0.0001,0.0748,0.0043
sma_distance_50,49,0,-0.1883,0.0002,0.0795,0.0069
candle_return,0,0,-4.5950,0.0000,6.2514,0.1697


## Save Dataset & Summary Report
- Persist the complete dataset with original OHLCV and all 17 engineered candidate features to `data/silver_features.csv`, reload, and verify schema.

In [11]:
output_cols = ["datetime_et", "open", "high", "low", "close", "volume"] + FEATURE_COLS
df[output_cols].to_csv(OUTPUT_FILE, index=False)

reloaded = pd.read_csv(OUTPUT_FILE)
assert reloaded.shape == df[output_cols].shape, "Shape mismatch upon reload!"
assert list(reloaded.columns) == output_cols, "Column mismatch upon reload!"

reloaded["datetime_et"] = pd.to_datetime(reloaded["datetime_et"], utc=True).dt.tz_convert("America/New_York")
file_size_mb = OUTPUT_FILE.stat().st_size / (1024**2)

final_report = pd.DataFrame([
    ("Input Dataset", f"{INPUT_FILE.name}"),
    ("Output Dataset", f"{OUTPUT_FILE.name}"),
    ("Total Rows", f"{len(reloaded):,}"),
    ("Complete Feature Rows (index >= 50)", f"{len(reloaded.dropna()):,}"),
    ("Original OHLCV Columns", "6 (datetime_et, open, high, low, close, volume)"),
    ("Engineered Candidate Features", f"{len(FEATURE_COLS)}"),
    ("Total Output Columns", f"{reloaded.shape[1]}"),
    ("Warmup Lookback NaNs", f"50 rows (initial lookback for volatility_50/sma_50)"),
    ("Infinite Values", "0"),
    ("File Size", f"{file_size_mb:.2f} MB"),
    ("Date Span", f"{reloaded['datetime_et'].min()} -> {reloaded['datetime_et'].max()}"),
    ("Pipeline Status", "Ready for Model Training & Validation (Notebook 06)")
], columns=["Metric", "Result"])

display(final_report)
display(reloaded[output_cols].head(3))

,Metric,Result
0,Input Dataset,silver_5m.csv
1,Output Dataset,silver_features.csv
2,Total Rows,"189,798"
3,Complete Feature Rows (index >= 50),"189,748"
4,Original OHLCV Columns,"6 (datetime_et, open, high, low, close, volume)"
5,Engineered Candidate Features,17
6,Total Output Columns,23
7,Warmup Lookback NaNs,50 rows (initial lookback for volatility_50/sm...
8,Infinite Values,0
9,File Size,66.31 MB


,datetime_et,open,high,low,close,volume,return_1,return_3,return_6,volatility_5,volatility_20,volatility_50,sma_distance_20,sma_distance_50,candle_return,candle_range_pct,upper_wick_ratio,lower_wick_ratio,relative_volume,volume_change,hour_sin,hour_cos,day_of_week
0,2023-11-01 02:10:00-04:00,22.75,22.79,22.745,22.770,229,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.087912,0.197628,0.444444,0.111111,0.0,0.000000,0.537300,0.843391,2
1,2023-11-01 02:15:00-04:00,22.77,22.79,22.765,22.785,84,0.065876,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.065876,0.109721,0.200000,0.200000,0.0,-63.318777,0.555570,0.831470,2
2,2023-11-01 02:20:00-04:00,22.78,22.78,22.770,22.775,59,-0.043889,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.021949,0.043908,0.000000,0.500000,0.0,-29.761905,0.573576,0.819152,2
